In [ ]:
!pip install -q -U transformers chromadb sentence-transformers "pillow<12.0"

!pip install -q --upgrade transformers

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)
print("Hugging Face login successful!")
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
from transformers.cache_utils import DynamicCache
from sentence_transformers import SentenceTransformer
import chromadb
from PIL import Image
import os
import shutil

if not hasattr(DynamicCache, 'seen_tokens'):
    @property
    def seen_tokens(self):
        return self.get_seq_length()
    DynamicCache.seen_tokens = seen_tokens

if not hasattr(DynamicCache, 'get_max_length'):
    def get_max_length(self):
    DynamicCache.get_max_length = get_max_length

if not hasattr(DynamicCache, 'get_usable_length'):
    def get_usable_length(self, new_seq_length, layer_idx=0):
        return self.get_seq_length()
    DynamicCache.get_usable_length = get_usable_length

print("Applied DynamicCache compatibility patches")
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
from transformers.cache_utils import DynamicCache
from sentence_transformers import SentenceTransformer
import chromadb
from PIL import Image
import os
import shutil
INPUT_DATA_PATH = "/kaggle/input/knowledge-dataset/"

READ_ONLY_DB_PATH = os.path.join(INPUT_DATA_PATH, "embedding_vectors")
IMG_PATH = os.path.join(INPUT_DATA_PATH, "all_images")

WRITABLE_DB_PATH = "/kaggle/working/my_vector_db"
print(f"Copying database from {READ_ONLY_DB_PATH} to {WRITABLE_DB_PATH}...")

if os.path.exists(WRITABLE_DB_PATH):
    print("Writable copy already exists. Removing old one.")
    shutil.rmtree(WRITABLE_DB_PATH)

shutil.copytree(READ_ONLY_DB_PATH, WRITABLE_DB_PATH)
print("Database copied successfully.")
print(f"Loading images relative to: {IMG_PATH}")

print("Clearing GPU memory...")
import gc
torch.cuda.empty_cache()
gc.collect()

print("Loading Phi-3-Vision model...")
device = "cuda" if torch.cuda.is_available() else "cpu"

phi3_processor = AutoProcessor.from_pretrained(
    "microsoft/Phi-3-vision-128k-instruct",
    trust_remote_code=True
)
config = AutoConfig.from_pretrained(
    "microsoft/Phi-3-vision-128k-instruct",
    trust_remote_code=True
)
config._attn_implementation = "eager"

phi3_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-vision-128k-instruct",
    config=config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=True
).to(device)

phi3_model.config.use_cache = False

print("Phi-3-Vision loaded.")

print("Loading text embedding model (google/embeddinggemma-300m) on CPU...")
text_embed_model = SentenceTransformer('google/embeddinggemma-300m', device="cpu")
print("Embedding model loaded.")

print("Loading vector database...")
client = chromadb.PersistentClient(path=WRITABLE_DB_PATH)
collection = client.get_collection(name="assam_disasters_multimodal")
print(f"Database loaded. Total items: {collection.count()}")
def analyze_image(image_path, analysis_prompt="Describe this image in detail."):
    """Analyzes a single image using Phi-3-Vision."""
    print(f"ANALYZING IMAGE: {image_path}")

    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        print(f"ERROR: Image file not found at {image_path}. Please check the path.")
        return
    except Exception as e:
        print(f"ERROR loading image: {e}")
        return

    prompt = f"<|user|>\n<|image_1|>\n{analysis_prompt}<|end|>\n<|assistant|>\n"

    print("Generating analysis with Phi-3-Vision...")

    inputs = phi3_processor(prompt, [image], return_tensors="pt").to(device)

    with torch.no_grad():
        generate_ids = phi3_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            eos_token_id=phi3_processor.tokenizer.eos_token_id,
            use_cache=False
        )
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
    answer = phi3_processor.batch_decode(
        generate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    print(answer)

import time
def run_multimodal_rag(question, n_results=3):
    start_time = time.time()

    print(f"\n--- New Question ---")
    print(f"Q: {question}\n")

    print("Embedding question...")
    text_embed_model.to(device)
    query_embedding = text_embed_model.encode(question).tolist()
    text_embed_model.to("cpu")
    torch.cuda.empty_cache()

    print("Retrieving relevant documents...")
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    if not results or not results['documents'][0]:
        print("We couldn't find any relevant documents in my knowledge base to answer that question.")
        return
    context_text_list = []
    context_image_paths = []

    documents = results['documents'][0]
    metadatas = results['metadatas'][0]

    MAX_TEXT_LENGTH = 1500
    total_length = 0

    for doc, meta in zip(documents, metadatas):
        if total_length + len(doc) < MAX_TEXT_LENGTH:
            context_text_list.append(doc)
            total_length += len(doc)

        image_paths_str = meta.get('image_paths', '')
        if image_paths_str:
            paths = image_paths_str.split(',')
            for p in paths:
                img_filename = os.path.basename(p)
                correct_path = os.path.join(IMG_PATH, img_filename)

                if os.path.exists(correct_path):
                    context_image_paths.append(correct_path)

    context_image_paths = sorted(list(set(context_image_paths)))

    print(f"Found {len(context_text_list)} text chunks and {len(context_image_paths)} unique images.")

    prompt = "Answer based on the context provided.\n\n"

    for i, text in enumerate(context_text_list):
        truncated = text[:400] + "..." if len(text) > 400 else text
        prompt += f"Context {i+1}: {truncated}\n\n"

    context_images = []
    MAX_IMAGES = 1
    for img_path in context_image_paths[:MAX_IMAGES]:
        try:
            img = Image.open(img_path).convert("RGB")
            context_images.append(img)
        except Exception as e:
            print(f"Warning: could not load image {img_path}: {e}")

    if context_images:
        prompt += f"<|image_1|>\n\n"

    prompt += f"Question: {question}\nAnswer:"

    full_prompt = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n"

    print("Generating answer with Phi-3-Vision...")

    torch.cuda.empty_cache()

    inputs = phi3_processor(full_prompt, context_images if context_images else None, return_tensors="pt")
    inputs = {k: v.to(phi3_model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

    with torch.no_grad():
        generate_ids = phi3_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=phi3_processor.tokenizer.eos_token_id,
            use_cache=False,
            pad_token_id=phi3_processor.tokenizer.eos_token_id
        )
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
    answer = phi3_processor.batch_decode(
        generate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    end_time = time.time()
    latency = end_time - start_time

    print("\n--- Answer ---")
    print(f"A: {answer}")
    print(f"\n[Latency: {latency:.2f} seconds]")

    print("\n--- Sources ---")
    for meta in metadatas:
        print(f"  - {meta['source_pdf']}, Page: {meta['page_number']}")
    torch.cuda.empty_cache()
import time
def analyze_image(image_path, analysis_prompt="Describe this image in detail."):
    start_time = time.time()

    print(f"\n============================\nANALYZING IMAGE: {image_path}\n============================")

    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        print(f"ERROR: Image file not found at {image_path}. Please check the path.")
        return
    except Exception as e:
        print(f"ERROR loading image: {e}")
        return
    prompt = f"<|user|>\n<|image_1|>\n{analysis_prompt}<|end|>\n<|assistant|>\n"

    print("Generating analysis with Phi-3-Vision...")

    inputs = phi3_processor(prompt, [image], return_tensors="pt")
    inputs = {k: v.to(phi3_model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

    with torch.no_grad():
        generate_ids = phi3_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            eos_token_id=phi3_processor.tokenizer.eos_token_id,
            use_cache=False,
            pad_token_id=phi3_processor.tokenizer.eos_token_id
        )

    generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
    answer = phi3_processor.batch_decode(
        generate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    end_time = time.time()
    latency = end_time - start_time

    print("\n=========== DESCRIPTION ===========\n")
    print(answer)
    print(f"\n[Latency: {latency:.2f} seconds]")

def run_analysis():
    print("--- Multimodal RAG & Analysis Tool ---")
    while True:
        print("\nOptions:")
        print(" [T] Text RAG")
        print(" [I] Image Analysis")
        print(" [Q] Quit")

        mode = input("Select mode (T/I/Q): ").strip().upper()

        if mode == 'Q':
            print("Exiting...")
            break

        elif mode == 'T':
            question = input("\nEnter your RAG query: ")
            if question:
                try:
                    n_results = 3
                    run_multimodal_rag(question, n_results=n_results)
                except Exception as e:
                    print(f"Error during RAG execution: {e}")

        elif mode == 'I':
            img_path = input("\nEnter full image path: ").strip()
            if img_path:
                analyze_image(img_path, analysis_prompt=prompt)
            else:
                print("No path provided.")

        else:
            print("Invalid selection. Please try again.")
run_analysis()
